<a href="https://colab.research.google.com/github/Ronald-Tuncar/LAB_13/blob/develop/LAB13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 13: Comparación del Desempeño de Modelos
### Nombre: Tuncar Andia Ronald

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Cargar dataset
df = pd.read_csv('/content/garments_worker_productivity.csv')

In [4]:
# Preprocesamiento inicial
df_clean = df.drop(columns=['date']).copy()
df_clean['target'] = np.where(df['actual_productivity'] < 0.5, 0, 1)
df_clean.drop(columns=['actual_productivity'], inplace=True)
df_clean['wip'].fillna(df_clean['wip'].median(), inplace=True)

df_clean.head()

<ipython-input-4-61310411>:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['wip'].fillna(df_clean['wip'].median(), inplace=True)


,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,target
0,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,1
1,Quarter1,finishing,Thursday,1,0.75,3.94,1039.0,960,0,0.0,0,0,8.0,1
2,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,1
3,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,1
4,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,1


In [5]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [6]:
# Columnas
categorical_cols = ['quarter', 'department', 'day']
numerical_cols = df_clean.drop(columns=categorical_cols + ['target']).columns.tolist()

In [7]:
# Preprocesador
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', OneHotEncoder(drop='first'), categorical_cols)
])


In [8]:
# Variables
X = df_clean.drop(columns='target')
y = df_clean['target']

In [9]:
# División
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)


b. Entrenamiento de modelos con distintos algoritmos

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [11]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': SVC(),
    'k-NN': KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier()
}

results = []
for name, model in models.items():
    model.fit(X_train_prep, y_train)
    y_pred = model.predict(X_test_prep)
    acc = accuracy_score(y_test, y_pred)
    results.append({'Modelo': name, 'Accuracy (default)': acc})

pd.DataFrame(results)

,Modelo,Accuracy (default)
0,Logistic Regression,0.879167
1,SVM,0.900000
2,k-NN,0.908333
3,Random Forest,0.904167
4,Naive Bayes,0.854167
5,Decision Tree,0.891667


c. Comparación de hiperparámetros con GridSearchCV

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train_prep, y_train)

print("Mejor Accuracy (k-NN):", grid_knn.best_score_)
print("Mejores parámetros:", grid_knn.best_params_)

Mejor Accuracy (k-NN): 0.8965205061082024
Mejores parámetros: {'metric': 'euclidean', 'n_neighbors': 7, 'weights': 'distance'}
